# 06 — SQL Cross-Validation

**RouteIQ — Delivery Operations Analytics**

**Business question:** Do the core KPIs computed independently in Python
(this notebook) reconcile exactly with the independently-built PostgreSQL
layer from Phase 3?

**Data used:** `data/cleaned/cleaned_delivery.csv` (43,648 rows), read-only.
The SQL-side figures below are **quoted, not recomputed**, from the
already-approved `reports/python_sql_cross_validation.md` /
`reports/sql_analysis_validation.md` — this notebook does not connect to
PostgreSQL or re-run any `sql/analysis/*.sql` file; it reproduces the
Python side of that existing, approved reconciliation.

**Analysis performed:** mirrors `python/analysis/05_cross_validation.py`'s
calculations line-for-line (same pandas expressions), computed fresh here.
This notebook does **not** call that module's `main()` — `main()` writes to
`output/eda_summary.json`, and this notebook must not modify any approved
output file, so every figure below is computed directly, read-only.


In [1]:
from _nb_theme import setup_paths, apply_theme, NAVY, TEAL, CORAL, NEUTRAL_SEQUENCE, annotate_bars, sample_size_caption
project_root = setup_paths()
apply_theme()

import sys
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from _common import load_cleaned_data

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
df = load_cleaned_data()
print(f"Loaded {len(df):,} rows, {df.shape[1]} columns from data/cleaned/cleaned_delivery.csv")


2026-08-17 14:29:08 | INFO     | routeiq_phase4 | Logging initialized. Log file: C:\Data Analyst Projects\RouteIQ Project\logs\phase1_pipeline_20260817_142908.log


2026-08-17 14:29:08 | INFO     | routeiq_phase4 | Loaded approved cleaned dataset: 43648 rows, 30 columns


Loaded 43,648 rows, 30 columns from data/cleaned/cleaned_delivery.csv


## 1. Core Row-Count and SLA KPIs


In [2]:
python_side = {
    "Total deliveries": int(len(df)),
    "Distinct Order_IDs": int(df["Order_ID"].nunique()),
    "Total SLA breaches": int(df["sla_breach_flag"].sum()),
    "Overall breach rate (%)": round(float(df["sla_breach_flag"].mean() * 100), 4),
}
sql_side = {
    "Total deliveries": 43_648,
    "Distinct Order_IDs": 43_648,
    "Total SLA breaches": 10_328,
    "Overall breach rate (%)": 23.6620,
}
core_kpi_df = pd.DataFrame({"SQL (Phase 3)": sql_side, "Python (this notebook)": python_side})
core_kpi_df["Match"] = core_kpi_df["SQL (Phase 3)"] == core_kpi_df["Python (this notebook)"]
core_kpi_df


,SQL (Phase 3),Python (this notebook),Match
Total deliveries,"43,648.0000","43,648.0000",True
Distinct Order_IDs,"43,648.0000","43,648.0000",True
Total SLA breaches,"10,328.0000","10,328.0000",True
Overall breach rate (%),23.6620,23.6620,True


**Key observation:** All four core KPIs match exactly between the
independently-built SQL layer and this notebook's fresh pandas
computation.


## 2. Breach Rate by Area


In [3]:
python_area = (df.groupby("Area", observed=True)["sla_breach_flag"].mean() * 100).round(4)
sql_area = pd.Series({"Metropolitian": 26.5000, "Urban": 14.3738, "Semi-Urban": 100.0000, "Other": 11.4437})
area_compare = pd.DataFrame({"SQL (%)": sql_area, "Python (%)": python_area})
area_compare["Match"] = area_compare["SQL (%)"] == area_compare["Python (%)"]
area_compare


,SQL (%),Python (%),Match
Metropolitian,26.5000,26.5000,True
Other,11.4437,11.4437,True
Semi-Urban,100.0000,100.0000,True
Urban,14.3738,14.3738,True


## 3. Category, Vehicle, Weather, Traffic Row Counts

**Analysis performed:** `value_counts()` per dimension, compared against
the SQL `GROUP BY ... COUNT(*)` figures already validated in
`reports/python_sql_cross_validation.md`.


In [4]:
sql_category = {
    "Electronics": 2843, "Books": 2817, "Jewelry": 2796, "Toys": 2773, "Snacks": 2767,
    "Skincare": 2766, "Outdoors": 2741, "Apparel": 2722, "Sports": 2711, "Grocery": 2688,
    "Pet Supplies": 2684, "Home": 2680, "Cosmetics": 2670, "Kitchen": 2667, "Clothing": 2662, "Shoes": 2661,
}
python_category = df["Category"].value_counts().to_dict()
category_mismatches = {k: (v, python_category.get(k)) for k, v in sql_category.items() if v != python_category.get(k)}
print(f"Category counts: {len(sql_category)} categories checked, {len(category_mismatches)} mismatches")

sql_vehicle = {"motorcycle": 25519, "scooter": 14607, "van": 3522}
python_vehicle = df["Vehicle"].value_counts().to_dict()
vehicle_mismatches = {k: (v, python_vehicle.get(k)) for k, v in sql_vehicle.items() if v != python_vehicle.get(k)}
print(f"Vehicle counts: {len(sql_vehicle)} types checked, {len(vehicle_mismatches)} mismatches")

sql_weather = {"Fog": 7440, "Stormy": 7374, "Cloudy": 7288, "Sandstorms": 7245, "Windy": 7223, "Sunny": 7078}
python_weather = df["Weather"].value_counts().to_dict()
weather_mismatches = {k: (v, python_weather.get(k)) for k, v in sql_weather.items() if v != python_weather.get(k)}
print(f"Weather counts: {len(sql_weather)} conditions checked, {len(weather_mismatches)} mismatches")

sql_traffic = {"Low": 14999, "Jam": 13725, "Medium": 10628, "High": 4296}
python_traffic = df["Traffic"].value_counts().to_dict()
traffic_mismatches = {k: (v, python_traffic.get(k)) for k, v in sql_traffic.items() if v != python_traffic.get(k)}
print(f"Traffic counts: {len(sql_traffic)} levels checked, {len(traffic_mismatches)} mismatches")


Category counts: 16 categories checked, 0 mismatches
Vehicle counts: 3 types checked, 0 mismatches
Weather counts: 6 conditions checked, 0 mismatches
Traffic counts: 4 levels checked, 0 mismatches


**Key observation:** 0 mismatches across all 16 categories, 3 vehicle
types, 6 weather conditions, and 4 traffic levels — every group-by row
count reconciles exactly between SQL and Python.


## 4. Weekday / Weekend and Correlation Coefficients


In [5]:
python_weekday_avg = round(float(df.loc[~df["is_weekend"], "Delivery_Time"].mean()), 4)
python_weekend_avg = round(float(df.loc[df["is_weekend"], "Delivery_Time"].mean()), 4)
weekday_weekend = pd.DataFrame({
    "SQL (2dp)": [124.90, 124.96, 31_627, 12_021],
    "Python (this notebook)": [python_weekday_avg, python_weekend_avg,
                                int((~df["is_weekend"]).sum()), int(df["is_weekend"].sum())],
}, index=["Weekday avg delivery time (min)", "Weekend avg delivery time (min)", "Weekday count", "Weekend count"])
weekday_weekend


,SQL (2dp),Python (this notebook)
Weekday avg delivery time (min),124.9000,124.8960
Weekend avg delivery time (min),124.9600,124.9631
Weekday count,"31,627.0000","31,627.0000"
Weekend count,"12,021.0000","12,021.0000"


In [6]:
valid_rating = df.loc[df["agent_rating_valid_flag"]]
valid_coords = df.loc[df["coordinates_valid_flag"]]
valid_age_rating = df.loc[df["agent_age_valid_flag"] & df["agent_rating_valid_flag"]]

python_corr = {
    "Agent rating vs. delivery time": round(float(stats.pearsonr(valid_rating["Agent_Rating"], valid_rating["Delivery_Time"])[0]), 4),
    "Distance vs. delivery time": round(float(stats.pearsonr(valid_coords["distance_km"], valid_coords["Delivery_Time"])[0]), 4),
    "Agent age vs. delivery time": round(float(stats.pearsonr(valid_age_rating["Agent_Age"], valid_age_rating["Delivery_Time"])[0]), 4),
    "Agent age vs. agent rating": round(float(stats.pearsonr(valid_age_rating["Agent_Age"], valid_age_rating["Agent_Rating"])[0]), 4),
}
sql_corr = {
    "Agent rating vs. delivery time": -0.3077,
    "Distance vs. delivery time": 0.2781,
    "Agent age vs. delivery time": 0.2585,
    "Agent age vs. agent rating": -0.1176,
}
corr_compare = pd.DataFrame({"SQL CORR()": sql_corr, "Python pearsonr": python_corr})
corr_compare["Match"] = corr_compare["SQL CORR()"] == corr_compare["Python pearsonr"]
corr_compare


,SQL CORR(),Python pearsonr,Match
Agent rating vs. delivery time,-0.3077,-0.3077,True
Distance vs. delivery time,0.2781,0.2781,True
Agent age vs. delivery time,0.2585,0.2585,True
Agent age vs. agent rating,-0.1176,-0.1176,True


**Key observation:** Weekday/weekend counts match exactly (31,627 + 12,021
= 43,648); average delivery times match SQL's 2-decimal rounding exactly.
All 4 correlation coefficients (agent rating, distance, agent age vs. time,
agent age vs. rating) match SQL's `CORR()` output exactly.

**Limitation:** None — 0 discrepancies across every figure checked in this
notebook, consistent with `reports/python_sql_cross_validation.md`'s
conclusion.


## 5. Metropolitian Breach-Volume Concentration (Pareto)

Cross-checks the Pareto figure used throughout notebooks 03-04 against
`sql/analysis/Q14`'s cumulative-share output.


In [7]:
m3 = importlib.import_module("03_correlation_and_pareto")
pareto_df = m3.pareto_ranking(df)
metro_share = pareto_df.loc[
    (pareto_df["dimension"] == "area") & (pareto_df["segment"] == "Metropolitian"),
    "cumulative_breach_share_pct"
].iloc[0]
print(f"Python-computed Metropolitian cumulative breach share: {metro_share}%")
print(f"SQL (Q14) documented figure: 83.7335%")
print("MATCH" if metro_share == 83.7335 else "MISMATCH")


Python-computed Metropolitian cumulative breach share: 83.7335%
SQL (Q14) documented figure: 83.7335%
MATCH


## Final Reconciliation Summary


In [8]:
final_check = pd.Series({
    "Total rows = 43,648": len(df) == 43_648,
    "Breach rate = 23.6620%": round(float(df["sla_breach_flag"].mean() * 100), 4) == 23.6620,
    "Category counts (16/16)": len(category_mismatches) == 0,
    "Vehicle counts (3/3)": len(vehicle_mismatches) == 0,
    "Weather counts (6/6)": len(weather_mismatches) == 0,
    "Traffic counts (4/4)": len(traffic_mismatches) == 0,
    "Correlation coefficients (4/4)": bool(corr_compare["Match"].all()),
    "Breach rate by area (4/4)": bool(area_compare["Match"].all()),
    "Metropolitian breach-volume concentration = 83.7335%": metro_share == 83.7335,
})
print("ALL CHECKS PASS" if final_check.all() else "DISCREPANCY FOUND")
final_check.to_frame("Pass")


ALL CHECKS PASS


,Pass
"Total rows = 43,648",True
Breach rate = 23.6620%,True
Category counts (16/16),True
Vehicle counts (3/3),True
Weather counts (6/6),True
Traffic counts (4/4),True
Correlation coefficients (4/4),True
Breach rate by area (4/4),True
Metropolitian breach-volume concentration = 83.7335%,True


**Conclusion:** 0 discrepancies found between the SQL (Phase 3) and Python
(Phase 4 / this notebook) layers across every figure checked — consistent
with `reports/python_sql_cross_validation.md`'s original conclusion. No
value on either side was adjusted, rounded differently, or recalculated to
force a match.

---

This is the final notebook in the RouteIQ JupyterLab analytical layer. See
`reports/jupyterlab_conversion_validation.md` for the full conversion and
validation summary across all 6 notebooks.
